### Load a trained model and predict GPP and ET

This notebook loads one trained model from the spatial nested cross-validation, rebuilds its test data (including the PRELES predictions) and plots predicted against observed GPP and ET for the held-out site.

**Before you start**
- Build the `preles_c` extension (README, section 2).
- Put the trained models under `checkpoints/` (your own training output, or the pretrained models from Zenodo). The expected layout is
  `checkpoints/preles/full/station_<length>/logs_spatial_<model type>/outer_station_<site>/`
  with `config_<k>_summary.json`, `models/best_model_<k>.pth` and `models/normalization.json`.
- The PRELES calibration for the model's training split must exist under `Data/calibrations/` (shipped for the CV designs in `configs/CV_splits/Paper_1_preles_ncvs/`).

The mixture-of-experts models (`logs_spatial_mixture`) use a different model class and are not covered here.

In [ ]:
import os
import json

import numpy as np
import matplotlib.pyplot as plt
import torch

from NN_framework import NN_module
from utilis_shorts import load_normalization
from process_model_calibration_manager import ProcessModelCalibrationManager
from calibration_dataloader_handoff import build_split_dataloader

%matplotlib inline

#### Choose a model
- `model_type`: `naive`, `bias`, `parallel`, `regularized` or `residual`
- `outer_fold`: the held-out (predicted) site: 5 Collelongo (Italy), 12 Hyytiälä (Finland), 14 Le Bray (France), 21 Sorø (Denmark)
- `length`: number of training years per site in the CV design (`len3`, `len6`, `len9`, `len12`)
- `draw`: which of the top HPO configurations to load (1 = best)

In [ ]:
model_type = "residual"
outer_fold = 12
length = "len12"
draw = 1

checkpoint_root = os.getenv("CHECKPOINT_PATH", "checkpoints")
fold_dir = os.path.join(checkpoint_root, "preles", "full", f"station_{length}",
                        f"logs_spatial_{model_type}", f"outer_station_{outer_fold}")
print(fold_dir)

#### Load the model
The summary file holds the full configuration the model was trained with, including its training and test site-years. The model runs on the CPU here.

In [ ]:
with open(os.path.join(fold_dir, f"config_{draw}_summary.json")) as f:
    model_config = json.load(f)[0]["config"]
model_config["training::cuda"] = False

model = NN_module(model_config)
saved = torch.load(os.path.join(fold_dir, "models", f"best_model_{draw}.pth"),
                   map_location="cpu", weights_only=False)
model.load_state_dict(saved.state_dict())
model.model.to("cpu").eval()

train_stations = model_config["data::train::station_id"]
test_stations = model_config["data::test::station_id"]
print("prediction type:", model_config["training::loss_physic_type"])
print("train:", train_stations)
print("test: ", test_stations)

#### Load the test data

The data are built the same way as in training:
1. PRELES runs with the parameters calibrated on this model's **training** split, on the continuous daily series, and its GPP/ET predictions are added to the data.
2. The inputs and targets are z-scored with the constants fitted on the training split (`models/normalization.json`). The test site's own statistics are never used.

The stored configuration contains the paths of the machine it was trained on, so they are replaced with paths in this repository.

In [ ]:
data_dir = os.path.join(os.getcwd(), "Data")
model_config["cfg::save"] = os.getcwd()
model_config["data::train::path_dir"] = data_dir
model_config["data::test::path_dir"] = data_dir
model_config["calibration::root"] = "calibrations"   # Data/calibrations

pm_manager = ProcessModelCalibrationManager(config=model_config, process_model="preles")
_, test_df, _ = pm_manager.load_fold_datasets(
    train_stations=train_stations,
    test_stations=test_stations,
    calibrate="require_existing",   # use the shipped calibration, never recalibrate
)

normalization = load_normalization(os.path.join(fold_dir, "models"))
test_loader = build_split_dataloader(model_config, test_df, split="test",
                                     process_model="preles", normalization=normalization)
test_df.head()

#### Predict
`parallel` models predict a correction that is added to the PRELES prediction; all other types predict the fluxes directly. Predictions and observations are transformed back to physical units with the training constants.

In [ ]:
prediction_type = model_config["training::loss_physic_type"]
targets = model_config["data::train::y_parameter"]   # ['GPP', 'ET']

preds, obs = [], []
with torch.no_grad():
    for batch in test_loader:
        X, y = batch[0].cpu(), batch[1].cpu()   # the loader may place tensors on a GPU
        pred = model.model(X)
        if prediction_type == "parallel":
            pred = pred + batch[2].cpu()
        preds.append(pred.numpy())
        obs.append(y.numpy())

dataset = test_loader.dataset
y_pred = dataset.denormalize_targets(np.concatenate(preds), normalization)
y_obs = dataset.denormalize_targets(np.concatenate(obs), normalization)

def nse(o, p):
    return 1 - np.sum((o - p) ** 2) / np.sum((o - o.mean()) ** 2)

for i, t in enumerate(targets):
    print(f"{t}: NSE = {nse(y_obs[:, i], y_pred[:, i]):.3f}")

#### Plot

In [ ]:
process_cols = [f"{t}_pred" for t in targets]   # PRELES predictions, physical units

fig, axes = plt.subplots(len(targets), 1, figsize=(14, 4 * len(targets)), sharex=True)
for i, (ax, t) in enumerate(zip(np.atleast_1d(axes), targets)):
    ax.plot(y_obs[:, i], label="observed", linewidth=1.2)
    ax.plot(y_pred[:, i], label=f"{model_type}", linewidth=1.2, alpha=0.85)
    ax.plot(test_df[process_cols[i]].to_numpy(), label="PRELES", linewidth=1.0, alpha=0.6)
    ax.set_ylabel(t)
    ax.grid(alpha=0.25)
    ax.legend(loc="upper right")
np.atleast_1d(axes)[-1].set_xlabel("day of test period")
fig.suptitle(f"{model_type}, held-out site {outer_fold}, {length}")
plt.tight_layout()
plt.show()